<div style="background-color: #f8fafc; border-left: 6px solid #7c3aed; padding: 20px 24px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.06); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
  <span style="font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 1.5px; color: #7c3aed;">AIML Honors &bull; Lab Assignment 03</span>
  <h1 style="margin: 8px 0 12px 0; color: #0f172a; font-size: 24px; font-weight: 700;">Relational Database Management & CRUD with SQLite</h1>
  <div style="font-size: 14px; color: #475569; line-height: 1.6;">
    <strong>Student Name:</strong> Pranjal Khadke &nbsp;|&nbsp; 
    <strong>Email:</strong> <a href="mailto:Pranjalckhadke@gmail.com" style="color: #7c3aed; text-decoration: none;">Pranjalckhadke@gmail.com</a> &nbsp;|&nbsp; 
    <strong>GitHub:</strong> <a href="https://github.com/PranjalKhadke" style="color: #7c3aed; text-decoration: none;">github.com/PranjalKhadke</a><br>
    <strong>Course:</strong> AIML Honors &bull; Sem III &nbsp;|&nbsp; 
    <strong>Topic:</strong> Relational Database Design, Parameterized SQL Queries & Transactions
  </div>
</div>

### 1. Aim & Theoretical Foundations

**Aim:** To engineer a relational database system using Python's native `sqlite3` interface, demonstrating DDL schema declaration, parameterized DML transactions, Primary Key constraints, and analytical SQL aggregations (`GROUP BY`).

#### Core Relational Database Principles:
1. **ACID Transaction Compliance:** Leveraging Python's `with sqlite3.connect(...) as conn:` context managers ensuring automatic atomic commits and rollbacks on runtime exceptions.
2. **SQL Injection Defense:** All queries pass inputs as tuple parameters (`?` placeholders), strictly isolating user data from executable SQL commands.
3. **B-Tree Primary Key Indexing:** `emp_id INTEGER PRIMARY KEY` functions as an alias to SQLite's internal 64-bit signed `ROWID`, yielding $O(1)$ point queries.

#### Table DDL Specification:
```sql
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    salary REAL NOT NULL CHECK(salary >= 0)
);
```


In [1]:
import sqlite3

DB_NAME = "employee.db"

def get_connection(db_name=DB_NAME):
    return sqlite3.connect(db_name)

def init_db(db_name=DB_NAME):
    with get_connection(db_name) as conn:
        cursor = conn.cursor()
        cursor.execute("DROP TABLE IF EXISTS employees")
        cursor.execute("""
            CREATE TABLE employees (
                emp_id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                department TEXT NOT NULL,
                salary REAL NOT NULL CHECK(salary >= 0)
            )
        """)
    print(f"[SQLITE ENGINE] Connected to '{db_name}'. Verified schema for table 'employees'.")

init_db()

[SQLITE ENGINE] Connected to 'employee.db'. Verified schema for table 'employees'.


In [2]:
def add_employee(emp_id, name, dept, salary, db_name=DB_NAME):
    try:
        with get_connection(db_name) as conn:
            cursor = conn.cursor()
            cursor.execute(
                "INSERT INTO employees (emp_id, name, department, salary) VALUES (?, ?, ?, ?)",
                (int(emp_id), str(name).strip(), str(dept).strip(), float(salary))
            )
        print(f"  [SUCCESS] Inserted record: {name} (ID: {emp_id}, Dept: {dept}, Salary: Rs. {float(salary):,.2f})")
        return True
    except sqlite3.IntegrityError:
        print(f"  [INTEGRITY ERROR] Unique constraint violated: Employee ID {emp_id} already exists.")
        return False

def get_all_employees(db_name=DB_NAME):
    with get_connection(db_name) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT emp_id, name, department, salary FROM employees ORDER BY emp_id ASC")
        rows = cursor.fetchall()
    if not rows:
        print("\n  [INFO] Database table 'employees' contains 0 records.\n")
        return []
    print("\n+--------+----------------------+------------------+----------------+")
    print("| Emp ID | Employee Name        | Department       | Salary (INR)   |")
    print("+--------+----------------------+------------------+----------------+")
    for r in rows:
        print(f"| {r[0]:<6} | {r[1]:<20} | {r[2]:<16} | Rs. {r[3]:>10,.2f} |")
    print("+--------+----------------------+------------------+----------------+\n")
    return rows

def get_employee_by_id(emp_id, db_name=DB_NAME):
    with get_connection(db_name) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT emp_id, name, department, salary FROM employees WHERE emp_id = ?", (int(emp_id),))
        row = cursor.fetchone()
    if row:
        print(f"  [RECORD] ID: {row[0]} | Name: {row[1]} | Dept: {row[2]} | Salary: Rs. {row[3]:,.2f}")
        return row
    print(f"  [NOT FOUND] No record matches Employee ID: {emp_id}.")
    return None

def update_employee(emp_id, name=None, dept=None, salary=None, db_name=DB_NAME):
    with get_connection(db_name) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name, department, salary FROM employees WHERE emp_id = ?", (int(emp_id),))
        current = cursor.fetchone()
        if not current:
            print(f"  [NOT FOUND] Unable to update: Employee ID {emp_id} not found in database.")
            return False
        new_name = str(name).strip() if name else current[0]
        new_dept = str(dept).strip() if dept else current[1]
        new_sal = float(salary) if salary is not None else current[2]
        cursor.execute(
            "UPDATE employees SET name = ?, department = ?, salary = ? WHERE emp_id = ?",
            (new_name, new_dept, new_sal, int(emp_id))
        )
    print(f"  [SUCCESS] Updated Employee ID {emp_id} -> Dept: {new_dept}, Salary: Rs. {new_sal:,.2f}")
    return True

def delete_employee(emp_id, db_name=DB_NAME):
    with get_connection(db_name) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM employees WHERE emp_id = ?", (int(emp_id),))
        deleted = cursor.rowcount > 0
    if deleted:
        print(f"  [SUCCESS] Deleted Employee ID {emp_id} from SQLite database.")
        return True
    print(f"  [NOT FOUND] Unable to delete: Employee ID {emp_id} does not exist.")
    return False

def get_department_stats(db_name=DB_NAME):
    with get_connection(db_name) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT department, COUNT(*), AVG(salary), MAX(salary)
            FROM employees
            GROUP BY department
            ORDER BY department ASC
        """)
        stats = cursor.fetchall()
    print("\n+------------------+-------------+----------------+----------------+")
    print("| Department       | Staff Count | Average Salary | Maximum Salary |")
    print("+------------------+-------------+----------------+----------------+")
    for s in stats:
        print(f"| {s[0]:<16} | {s[1]:<11} | Rs. {s[2]:>10,.2f} | Rs. {s[3]:>10,.2f} |")
    print("+------------------+-------------+----------------+----------------+\n")
    return stats

In [3]:
# Populating employee database with sample records
add_employee(201, "Pranjal Khadke", "AI Research", 88000)
add_employee(202, "Tanvi Joshi", "Data Science", 79000)
add_employee(203, "Rahul Mehta", "Engineering", 72000)
add_employee(204, "Sneha Roy", "Data Science", 84000)
add_employee(205, "Aditya Nair", "Engineering", 69000)

  [SUCCESS] Inserted record: Pranjal Khadke (ID: 201, Dept: AI Research, Salary: Rs. 88,000.00)
  [SUCCESS] Inserted record: Tanvi Joshi (ID: 202, Dept: Data Science, Salary: Rs. 79,000.00)
  [SUCCESS] Inserted record: Rahul Mehta (ID: 203, Dept: Engineering, Salary: Rs. 72,000.00)
  [SUCCESS] Inserted record: Sneha Roy (ID: 204, Dept: Data Science, Salary: Rs. 84,000.00)
  [SUCCESS] Inserted record: Aditya Nair (ID: 205, Dept: Engineering, Salary: Rs. 69,000.00)


In [4]:
# Querying all records via SELECT statement
get_all_employees()


+--------+----------------------+------------------+----------------+
| Emp ID | Employee Name        | Department       | Salary (INR)   |
+--------+----------------------+------------------+----------------+
| 201    | Pranjal Khadke       | AI Research      | Rs.  88,000.00 |
| 202    | Tanvi Joshi          | Data Science     | Rs.  79,000.00 |
| 203    | Rahul Mehta          | Engineering      | Rs.  72,000.00 |
| 204    | Sneha Roy            | Data Science     | Rs.  84,000.00 |
| 205    | Aditya Nair          | Engineering      | Rs.  69,000.00 |
+--------+----------------------+------------------+----------------+



In [5]:
# Point query: Search employee by primary key (ID: 201)
get_employee_by_id(201)

  [RECORD] ID: 201 | Name: Pranjal Khadke | Dept: AI Research | Salary: Rs. 88,000.00


In [6]:
# SQL Aggregations: Department-wise headcount and compensation analysis
get_department_stats()


+------------------+-------------+----------------+----------------+
| Department       | Staff Count | Average Salary | Maximum Salary |
+------------------+-------------+----------------+----------------+
| AI Research      | 1           | Rs.  88,000.00 | Rs.  88,000.00 |
| Data Science     | 2           | Rs.  81,500.00 | Rs.  84,000.00 |
| Engineering      | 2           | Rs.  70,500.00 | Rs.  72,000.00 |
+------------------+-------------+----------------+----------------+



In [7]:
# Executing parameterized UPDATE transaction
update_employee(203, dept="AI Research", salary=80000)
get_employee_by_id(203)

  [SUCCESS] Updated Employee ID 203 -> Dept: AI Research, Salary: Rs. 80,000.00
  [RECORD] ID: 203 | Name: Rahul Mehta | Dept: AI Research | Salary: Rs. 80,000.00


In [8]:
# Executing DELETE transaction and verifying table state
delete_employee(205)
get_all_employees()

  [SUCCESS] Deleted Employee ID 205 from SQLite database.

+--------+----------------------+------------------+----------------+
| Emp ID | Employee Name        | Department       | Salary (INR)   |
+--------+----------------------+------------------+----------------+
| 201    | Pranjal Khadke       | AI Research      | Rs.  88,000.00 |
| 202    | Tanvi Joshi          | Data Science     | Rs.  79,000.00 |
| 203    | Rahul Mehta          | AI Research      | Rs.  80,000.00 |
| 204    | Sneha Roy            | Data Science     | Rs.  84,000.00 |
+--------+----------------------+------------------+----------------+



In [9]:
# Verifying database constraint integrity
add_employee(201, "Duplicate Test", "Operations", 50000)

  [INTEGRITY ERROR] Unique constraint violated: Employee ID 201 already exists.


<div style="background-color: #f0fdf4; border: 1px solid #bbf7d0; border-left: 5px solid #16a34a; padding: 16px 20px; border-radius: 6px; font-family: sans-serif;">
  <h4 style="margin: 0 0 6px 0; color: #15803d; font-size: 15px; font-weight: 700;">Conclusion & Experimental Results</h4>
  <p style="margin: 0; color: #166534; font-size: 13.5px; line-height: 1.6;">
    The SQLite relational architecture significantly outperforms flat CSV storage by enforcing hardware-level constraints (UNIQUE, CHECK) and supporting multi-record SQL aggregations (<code>COUNT(*)</code>, <code>AVG()</code>, <code>MAX()</code>). Parameterized bindings successfully shielded the database against syntax anomalies, while context managers ensured zero database connection leaks.
  </p>
</div>